# Clean test on a free Google computer: score the reports with Llama 3.2

This notebook scores the same 525 earnings reports with **Llama 3.2 (3 billion parameters)**, a model whose
training data ends in December 2023, on a free Google Colab machine with a graphics card. It takes about
20 to 40 minutes and uses no Claude account.

**Before you start:** in the menu choose **Runtime → Change runtime type → T4 GPU → Save**.

Then run each cell in order with the ▶ button (or **Runtime → Run all**). The upload cell will ask you to choose
`colab_upload.zip` from your project folder. The last cell downloads `colab_results.zip`.

## 1. Check that a graphics card is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No graphics card: choose Runtime > Change runtime type > T4 GPU, then run this cell again"

## 2. Install Ollama, start it, and download Llama 3.2 (about 2 minutes)

In [ ]:
!sudo apt-get -qq install -y pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time, requests
server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("Ollama is running")
        break
    except Exception:
        time.sleep(1)
!ollama pull llama3.2:3b

## 3. Download the project code from GitHub

In [ ]:
%cd /content
!rm -rf llm-earnings-signal
!git clone -q https://github.com/pooja003-cloud/llm-earnings-signal.git
%cd /content/llm-earnings-signal
!pip install -q -e . 2>&1 | tail -1
!test -f config/llama.yaml && echo "Project ready" || echo "config/llama.yaml missing: push the latest version to GitHub first"

## 4. Upload `colab_upload.zip` from your project folder

It holds the 525 reports and the answers already scored on your Mac, so those are skipped.

In [ ]:
from google.colab import files
uploaded = files.upload()
!unzip -oq colab_upload.zip && echo "Reports in data/raw/docs:" && find data/raw/docs -name '*.txt' | wc -l

## 5. Score the reports (about 20 to 40 minutes)

It prints progress every 50 reports. If the session disconnects, run steps 2 and 5 again: answers are saved as they come in, but only within this session, so download (step 7) before closing.

In [ ]:
!earnsig --config config/llama.yaml score

## 6. Score 40 reports a second time to check consistency (about 2 minutes)

In [ ]:
!earnsig --config config/llama.yaml consistency

## 7. Download the results

Put `colab_results.zip` in your project folder on your Mac and follow the steps in the README (or the chat).

In [ ]:
!zip -q colab_results.zip data/llm_cache.jsonl data/scores_llama.csv results/llama/consistency.json && echo "Ready to download"
files.download("colab_results.zip")